# Severstal: frozen ABC ensemble

E07-r01: evaluate the previously selected A29/B11/C29 ensemble on Kaggle. No training or threshold search. The `area` and `soft` branches are declared before scores are observed. Private input weights; Internet OFF; GPU ON.


In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
BRANCH = "area"  # second predeclared submission: "soft"
roots = list(Path("/kaggle/input").rglob("manifest.json"))
bundles = [p.parent for p in roots if (p.parent / "A.pt").exists() and (p.parent / "C.pt").exists()]
assert len(bundles) == 1, f"Attach the frozen ABC model dataset: {bundles}"
bundle = bundles[0]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", "--find-links", str(bundle / "wheels"), "segmentation-models-pytorch==0.5.0", "timm==1.0.22"], check=True)
sys.path.insert(0, str(bundle))


In [ ]:
from severstal.kaggle_infer import run
competition = Path("/kaggle/input/severstal-steel-defect-detection")
assert (competition / "test_images").is_dir(), "Attach the official competition data"
result = run(bundle, competition / "test_images", Path("/kaggle/working") / f"predictions_{BRANCH}", (BRANCH,))
shutil.copyfile(Path("/kaggle/working") / f"predictions_{BRANCH}" / f"submission_{BRANCH}.csv", "/kaggle/working/submission.csv")
print({"branch": BRANCH, "images": result["images"], "rows": result["rows_per_submission"], "seconds": result["seconds"]})


The output is a prediction file, not a score. Submit the committed notebook to the competition, then record the returned Public/Private score and submission ID. Test image names are read at runtime so the same code covers hidden evaluation images.
